In [1]:
# Cell 1: install (run in a notebook cell or terminal)
!pip install -U "transformers[torch]" "torch" "ipywidgets"
# If you need sentencepiece for some tokenizers:
!pip install -U sentencepiece


   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.3 MB 2.6 MB/s eta 0:00:04
   ----- ---------------------------------- 1.6/11.3 MB 2.6 MB/s eta 0:00:04
   ------- -------------------------------- 2.1/11.3 MB 2.7 MB/s eta 0:00:04
   --------- ------------------------------ 2.6/11.3 MB 2.6 MB/s eta 0:00:04
   ----------- ---------------------------- 3.1/11.3 MB 2.7 MB/s eta 0:00:04
   ------------- -------------------------- 3.7/11.3 MB 2.7 MB/s eta 0:00:03
   -------------- ------------------------- 4.2/11.3 MB 2.6 MB/s eta 0:00:03
   ---------------- ----------------------- 4.7/11.3 MB 2.7 MB/s eta 0:00:03
   ------------------- -------------------- 5.5/11.3 MB 2.6 MB/s eta 0:00:03
   --------------------- ------------------ 6.0/11.3 MB 2.7 MB/s eta 0:00:02
   ----------------------- ---------------- 6.6/11.3 MB 2.6 MB/s eta 0:00:02
   ----------

In [2]:
# Cell 2: imports and secure HF token helper
import os
import getpass
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from functools import lru_cache
from typing import Tuple

# Optional UI lib (used later)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_IPYWIDGETS = True
except Exception:
    HAS_IPYWIDGETS = False

def get_hf_token(prompt_if_missing: bool = True) -> str:
    """
    Read HF token from environment HUGGINGFACEHUB_API_TOKEN, otherwise prompt (not echoed).
    Returns token string or empty string.
    """
    token = os.environ.get("HUGGINGFACEHUB_API_TOKEN") or os.environ.get("HF_TOKEN") or ""
    if not token and prompt_if_missing:
        # Use getpass so token is not echoed/visible
        token = getpass.getpass("Enter your Hugging Face token (input hidden; press Enter to skip): ").strip()
    return token


In [3]:
# Cell 3: model/tokenizer loader with caching
@lru_cache(maxsize=2)
def load_tokenizer_and_model(model_name: str, hf_token: str = "", trust_remote_code: bool = False) -> Tuple:
    """
    Loads and returns (tokenizer, model, device). Cached in-memory so repeated calls are fast.
    - model_name: HF repo id like "meta-llama/Llama-3.2-1B"
    - hf_token: Hugging Face token string or empty
    - trust_remote_code: True for repos that require remote code
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    kwargs = {}
    if hf_token:
        kwargs["use_auth_token"] = hf_token
    if trust_remote_code:
        kwargs["trust_remote_code"] = True

    print(f"Loading tokenizer and model '{model_name}' to {device} (this may take a while)...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, **kwargs)
    model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    model.to(device)
    model.eval()
    print("Model loaded.")
    return tokenizer, model, device


In [4]:
# Cell 4: generation function
def generate_text(tokenizer, model, device, prompt: str, approx_words: int,
                  temperature: float = 0.7, top_p: float = 0.9, do_sample: bool = True) -> str:
    """
    Generate text with a causal LM.
    approx_words: desired words — converted to tokens with a conservative multiplier.
    Returns generated text (only the new content, prompt stripped).
    """
    # conservative token estimate: ~1.5 tokens/word
    estimated_new_tokens = max(16, int(approx_words * 1.5))

    # Respect tokenizer/model max length if available
    model_max_length = getattr(tokenizer, "model_max_length", None)
    if model_max_length and model_max_length > 0:
        if estimated_new_tokens > model_max_length - 32:
            estimated_new_tokens = max(16, model_max_length - 32)

    # Encode prompt
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    # Build generation config
    gen_cfg = GenerationConfig(
        max_new_tokens=estimated_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id
    )

    with torch.no_grad():
        outputs = model.generate(**inputs, generation_config=gen_cfg)

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt prefix if present
    if decoded.startswith(prompt):
        decoded = decoded[len(prompt):].strip()
    return decoded


In [5]:
# Cell 5: prompt template helpers
def build_prompt(topic: str, approx_words: int, style: str) -> str:
    # fix spelling and be explicit
    return f"Write a {approx_words}-word blog post on the topic '{topic}' in a {style} style.\n\nBlog:\n"


In [6]:
# Cell 5: prompt template helpers
def build_prompt(topic: str, approx_words: int, style: str) -> str:
    # fix spelling and be explicit
    return f"Write a {approx_words}-word blog post on the topic '{topic}' in a {style} style.\n\nBlog:\n"


In [ ]:
# Cell 6: Interactive UI or simple prompt-run flow
def run_interactive():
    hf_token = get_hf_token(prompt_if_missing=False)  # check env first
    # Ask for token if not present
    if not hf_token:
        print("No HF token found in environment. If you plan to load gated models, you'll be prompted later.")

    model_name_w = widgets.Text(value="meta-llama/Llama-3.2-1B", description="Model:")
    topic_w = widgets.Text(value="Artificial Intelligence in 2025", description="Topic:")
    words_w = widgets.IntSlider(value=500, min=50, max=3000, step=50, description="Words:")
    style_w = widgets.Dropdown(options=["Informative", "Conversational", "Technical", "Casual"], value="Informative", description="Style:")
    temp_w = widgets.FloatSlider(value=0.7, min=0.0, max=1.5, step=0.05, description="Temp:")
    top_p_w = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="Top-p:")
    trust_w = widgets.Checkbox(value=False, description="trust_remote_code")
    token_w = widgets.Password(description="HF token (opt)", placeholder="Leave blank to use env var")
    btn = widgets.Button(description="Load model & Generate", button_style="primary")
    out = widgets.Output(layout={'border': '1px solid black'})

    controls = widgets.VBox([model_name_w, topic_w, words_w, style_w, temp_w, top_p_w, trust_w, token_w, btn])
    display(widgets.HBox([controls, out]))

    def on_click(b):
        with out:
            clear_output()
            # Prefer the explicit token input; if empty, use env var
            token = token_w.value.strip() or os.environ.get("HUGGINGFACEHUB_API_TOKEN", "")
            model_name = model_name_w.value.strip()
            topic = topic_w.value.strip()
            words = int(words_w.value)
            style = style_w.value
            temp = float(temp_w.value)
            top_p = float(top_p_w.value)
            trust_remote_code = bool(trust_w.value)

            if not token:
                print("No HF token provided. If the model requires authentication, loading will fail.")
            try:
                tokenizer, model, device = load_tokenizer_and_model(model_name, hf_token=token, trust_remote_code=trust_remote_code)
            except Exception as e:
                print("Model loading failed:", e)
                return

            prompt = build_prompt(topic, words, style)
            print("Prompt:\n", prompt)
            print("\nGenerating... (this may take some time)")
            try:
                generated = generate_text(tokenizer, model, device, prompt, words, temperature=temp, top_p=top_p, do_sample=True)
            except Exception as e:
                print("Generation failed:", e)
                return

            print("\n--- Generated text ---\n")
            print(generated)
            # Offer to save
            fname = f"blog_{topic.replace(' ', '_')[:40]}.txt"
            with open(fname, "w", encoding="utf-8") as f:
                f.write(generated)
            print(f"\nSaved output to ./{fname}")

    btn.on_click(on_click)

# Run the interactive UI if ipywidgets is available; otherwise do a simple input flow
if HAS_IPYWIDGETS:
    run_interactive()
else:
    # fallback: simple sequential prompts
    token = get_hf_token(prompt_if_missing=True)
    model_name = input("Model repo (eg. meta-llama/Llama-3.2-1B): ").strip() or "meta-llama/Llama-3.2-1B"
    topic = input("Topic: ").strip() or "Artificial Intelligence in 2025"
    words = int(input("Approx number of words (e.g. 500): ").strip() or "500")
    style = input("Style (Informative/Conversational/Technical/Casual): ").strip() or "Informative"
    trust = input("trust_remote_code? (y/N): ").strip().lower().startswith("y")

    tokenizer, model, device = load_tokenizer_and_model(model_name, hf_token=token, trust_remote_code=trust)
    prompt = build_prompt(topic, words, style)
    print("Prompt:\n", prompt)
    print("\nGenerating...\n")
    output = generate_text(tokenizer, model, device, prompt, words)
    print(output)
    with open("blog_output.txt", "w", encoding="utf-8") as f:
        f.write(output)
    print("Saved to blog_output.txt")
